# AI Lecture Tutor — Kaggle instructor demo

Run these cells in order in a Kaggle Notebook with Internet enabled and a GPU accelerator selected if one is available. The last cell prints the one temporary app link to share with the instructor. Keep this notebook session running during the demo. The link is open to anyone who has it; it does not ask the instructor for a password.

This notebook uses a Cloudflare Quick Tunnel for a short test. Kaggle runtime availability and tunnel connectivity are not guaranteed.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time, secrets, urllib.request

# Set this to the public GitHub URL for the project repository.
REPO_URL = 'https://github.com/YOUR-USERNAME/YOUR-REPOSITORY.git'
REPO_DIR = Path('/kaggle/working/ai-lecture-tutor')
if 'YOUR-USERNAME' in REPO_URL or 'YOUR-REPOSITORY' in REPO_URL:
    raise ValueError('Edit REPO_URL above to your project’s public GitHub URL, then rerun this cell.')
if not (REPO_DIR / 'backend' / 'api.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Project ready at', REPO_DIR)
print('GPU detected:', __import__('torch').cuda.is_available())

In [ ]:
%pip install -q -r backend/requirements.txt -r frontend/requirements.txt

In [ ]:
import json, platform, urllib.request

# Download the current Linux x86_64 cloudflared release binary.
release = json.load(urllib.request.urlopen('https://api.github.com/repos/cloudflare/cloudflared/releases/latest'))
asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
cloudflared_path = Path('/kaggle/working/cloudflared')
urllib.request.urlretrieve(asset['browser_download_url'], cloudflared_path)
cloudflared_path.chmod(0o755)
print('Installed cloudflared', release['tag_name'])

In [ ]:
import torch
from backend.config import MODEL_ID
from backend.llm.runtime import runtime

# Internal service credential only; it is never shown to the instructor.
os.environ['BACKEND_SECRET'] = secrets.token_urlsafe(32)
os.environ['BACKEND_URL'] = 'http://127.0.0.1:8000'
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Loading and answering may be very slow on CPU.')
print('Loading', MODEL_ID, 'and the sentence embedding model; this first run downloads several GB.')
runtime.load()
print('Models ready on', runtime.device)

In [ ]:
api = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'backend.api:app', '--host', '127.0.0.1', '--port', '8000'], cwd=str(REPO_DIR), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
            print('API ready:', response.read().decode())
            break
    except Exception:
        if api.poll() is not None:
            raise RuntimeError('FastAPI stopped during startup.')
        time.sleep(1)
else:
    raise TimeoutError('API did not become ready within 90 seconds.')

streamlit = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'frontend/app.py', '--server.address=127.0.0.1', '--server.port=8501', '--server.headless=true', '--browser.gatherUsageStats=false'], cwd=str(REPO_DIR), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=2) as response:
            print('Streamlit ready:', response.status)
            break
    except Exception:
        if streamlit.poll() is not None:
            raise RuntimeError('Streamlit stopped during startup.')
        time.sleep(1)
else:
    raise TimeoutError('Streamlit did not become ready within 60 seconds.')

In [ ]:
tunnel_log = open('/kaggle/working/cloudflared.log', 'w')
tunnel = subprocess.Popen([str(cloudflared_path), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8501'], stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(1)
    tunnel_log.flush()
    log_text = Path('/kaggle/working/cloudflared.log').read_text(errors='replace')
    import re
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        public_url = match.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError('Cloudflare tunnel exited. Inspect /kaggle/working/cloudflared.log.')
if not public_url:
    raise TimeoutError('No temporary app URL appeared. Inspect /kaggle/working/cloudflared.log.')
print('APP LINK FOR THE INSTRUCTOR:', public_url)
print('Leave this notebook session running. Stop the tunnel and app processes when the test is over.')